# MCP & LLM-Anbindung – Intelligente Fabrikagenten

**Internet of Production** · Prof. Dr.-Ing. U. Triltsch · Ostfalia Hochschule für angewandte Wissenschaften

---

## Lernziele

Nach Abschluss dieses Notebooks sind Sie in der Lage:

1. Das **MCP-Protokoll** (Model Context Protocol) auf Architektur- und Nachrichtenebene zu erläutern.
2. Einen funktionsfähigen **MCP-Server** zu implementieren, der Fabriktelemetrie als Tool exponiert.
3. Einen **MCP-Client** zu schreiben, der Tools aufruft und Antworten verarbeitet.
4. Ein **LLM als Entscheidungsinstanz** in einen LangGraph-Agenten zu integrieren.
5. Den Unterschied zwischen **regelbasierter** und **LLM-gestützter** Koordinationslogik am selben Szenario zu demonstrieren.

---

## Einordnung in die Vorlesungsreihe

| Notebook | Paradigma | Entscheidungslogik | Konnektivität |
|---|---|---|---|
| `W1` / `W2` Reaktiv / Kommunikativ | Sense–Decide–Act | Regelbasiert | OPC UA (direkt) |
| `01_LangGraph_Agents_solution.ipynb` | Stateful Graph | Regelbasiert | Simulierter Zustand |
| **`02_MCP_LLM_solution.ipynb` (dieses Notebook)** | **Stateful Graph** | **LLM-gestützt** | **MCP → OPC UA / REST** |

Das Fabrikszenario bleibt identisch: Maschinen erhitzen sich, ein Koordinatoragent
entscheidet über Reparaturen, ein Wartungsagent führt sie aus. Neu ist:
- Der Koordinator verwendet ein **LLM** statt hartcodierter Regeln.
- Die Telemetriedaten kommen über **MCP** statt aus einem simulierten Dict.

---

## Struktur dieses Notebooks

```
Teil 1 – MCP-Protokoll:       Architektur, Nachrichtenformat, Tool-Schema
Teil 2 – MCP-Server:          Implementierung mit FastAPI + mcp-Bibliothek
Teil 3 – MCP-Client:          Tool-Discovery, Tool-Call, Antwortverarbeitung
Teil 4 – LLM als Agent:       Stub-Implementierung, echter API-Aufruf (Erweiterung)
Teil 5 – Vollintegration:     LangGraph + MCP + LLM im Fabrikszenario
Teil 6 – Vergleich:           Regelbasiert vs. LLM-gestützt
```

---
## Installation

In [ ]:
# Kernabhängigkeiten
!pip install -q "langgraph>=0.2" "langchain-core>=0.2"
!pip install -q "fastapi" "uvicorn" "httpx" "requests"

# MCP Python SDK (offizielle Anthropic-Implementierung)
!pip install -q "mcp>=1.0"

# Für die echte LLM-Anbindung (Teil 4, Erweiterung)
#Alternative: 
# für Claude
#!pip install -q "langchain-anthropic>=0.1"  
 # für GPT-4o
!pip install -q "langchain-openai>=0.1" 

print("Installation abgeschlossen.")

In [ ]:
import json
import random
import threading
import time
import textwrap
from typing import TypedDict, Dict, List, Literal, Any, Optional

import requests
import uvicorn
from fastapi import FastAPI
from pydantic import BaseModel

from langgraph.graph import StateGraph, END

print("Alle Imports erfolgreich.")

---
# Teil 1 – MCP-Protokoll: Architektur und Nachrichtenformat

## 1.1 Was ist MCP und warum existiert es?

Vor MCP musste jede LLM-Anwendung ihre eigenen Integrationscode für jedes externe
System schreiben: eigene HTTP-Clients, eigene Authentifizierungslogik, eigene
Fehlerbehandlung. Mit wachsender Anzahl von Tools entstand ein **N×M-Integrationsproblem**.

```
OHNE MCP (N×M-Integrationen)        MIT MCP (N+M-Integrationen)

Agent A ──► OPC UA (custom)          Agent A ──► MCP-Client ──► MCP-Server: OPC UA
Agent A ──► REST API (custom)                                ──► MCP-Server: REST
Agent A ──► DB (custom)                                      ──► MCP-Server: DB
Agent B ──► OPC UA (custom)          Agent B ──► MCP-Client ──► (selbe Server)
Agent B ──► REST API (custom)
```

MCP löst dieses Problem durch **Standardisierung auf Protokollebene** — analog zu
wie HTTP das Web standardisiert hat.

## 1.2 Architektur: Host, Client, Server

```
┌─────────────────────────────────────────────┐
│  HOST (LLM-Anwendung / Agent-Framework)     │
│                                             │
│  ┌─────────────┐   ┌─────────────────────┐  │
│  │  LLM        │◄──│  MCP-Client A       │  │
│  │ (Reasoning) │   │  (OPC UA-Adapter)   │  │
│  └─────────────┘   └────────┬────────────┘  │
│                             │ JSON-RPC 2.0  │
└─────────────────────────────┼───────────────┘
                              │
              ┌───────────────▼──────────────┐
              │  MCP-SERVER                  │
              │  (läuft separat / remote)    │
              │                              │
              │  Tools:                      │
              │  - read_temperature(id)      │
              │  - write_temperature(id, v)  │
              │  - list_machines()           │
              └───────────────┬──────────────┘
                              │
                    ┌─────────▼──────────┐
                    │  OPC UA / REST /   │
                    │  Datenbank / SPS   │
                    └────────────────────┘
```

**Drei Rollen im MCP-Ökosystem:**

- **Host**: Die LLM-Anwendung (z. B. LangGraph-Graph, Claude Desktop). Enthält den MCP-Client.
- **MCP-Client**: Sprachbibliothek im Host, die das MCP-Protokoll implementiert. Verwaltet Verbindungen zu MCP-Servern.
- **MCP-Server**: Exponiert *Tools*, *Resources* oder *Prompts* über das MCP-Protokoll. Kapselt die Zugriffslogik auf das dahinterliegende System.

## 1.3 Nachrichtenformat: JSON-RPC 2.0

MCP basiert auf **JSON-RPC 2.0** — einem leichtgewichtigen Remote-Procedure-Call-Protokoll.
Alle Nachrichten sind JSON-Objekte mit festgelegter Struktur.

In [ ]:
# ── Beispiele für MCP-Nachrichten (JSON-RPC 2.0) ─────────────────────────────
# Diese Nachrichten werden normalerweise von der MCP-Bibliothek erzeugt.
# Das manuelle Inspizieren hilft, das Protokoll zu verstehen.

# 1. Tool-Discovery: Client fragt, welche Tools der Server anbietet
anfrage_tools_liste = {
    "jsonrpc": "2.0",
    "method": "tools/list",
    "params": {},
    "id": 1
}

# 2. Antwort des Servers mit Tool-Definitionen (Schema)
antwort_tools_liste = {
    "jsonrpc": "2.0",
    "result": {
        "tools": [
            {
                "name": "read_temperature",
                "description": "Liest die aktuelle Temperatur einer Maschine in °C.",
                "inputSchema": {
                    "type": "object",
                    "properties": {
                        "machine_id": {
                            "type": "string",
                            "description": "Maschinenkennung, z. B. 'M01'"
                        }
                    },
                    "required": ["machine_id"]
                }
            },
            {
                "name": "list_machines",
                "description": "Gibt eine Liste aller verfügbaren Maschinen zurück.",
                "inputSchema": {
                    "type": "object",
                    "properties": {}
                }
            }
        ]
    },
    "id": 1
}

# 3. Tool-Aufruf: Client ruft read_temperature für M03 auf
anfrage_tool_aufruf = {
    "jsonrpc": "2.0",
    "method": "tools/call",
    "params": {
        "name": "read_temperature",
        "arguments": {"machine_id": "M03"}
    },
    "id": 2
}

# 4. Antwort des Servers
antwort_tool_aufruf = {
    "jsonrpc": "2.0",
    "result": {
        "content": [
            {
                "type": "text",
                "text": json.dumps({"machine_id": "M03", "temperature": 74.2, "unit": "°C", "state": "HOT"})
            }
        ]
    },
    "id": 2
}

print("=== MCP-Nachrichtenbeispiele ===")
print("\n1. Tool-Discovery-Anfrage (Client → Server):")
print(json.dumps(anfrage_tools_liste, indent=2, ensure_ascii=False))
print("\n2. Tool-Aufruf (Client → Server):")
print(json.dumps(anfrage_tool_aufruf, indent=2, ensure_ascii=False))
print("\n3. Server-Antwort:")
print(json.dumps(antwort_tool_aufruf, indent=2, ensure_ascii=False))

## 1.4 MCP-Primitive: Tools, Resources, Prompts

MCP definiert drei Typen von Primitiven, die ein Server exponieren kann:

| Primitiv | Zweck | Industriebeispiel | Steuerung durch |
|---|---|---|---|
| **Tool** | Ausführbare Funktion mit Seiteneffekten | `read_temperature()`, `trigger_alarm()` | LLM (model-controlled) |
| **Resource** | Lesbare Datenquelle (kein Seiteneffekt) | OPC UA-Adressraum, Sensorzeitreihe | Anwendung (app-controlled) |
| **Prompt** | Wiederverwendbare Prompt-Vorlage | Diagnose-Prompt für HOT-Zustand | Benutzer (user-controlled) |

In diesem Notebook konzentrieren wir uns auf **Tools**, da sie die häufigste
Integrationsmethode für industrielle Agentensysteme sind.

**Sicherheitshinweis**: Da Tools Seiteneffekte haben können
(Maschinenstopp, Alarmauslösung), empfiehlt MCP eine explizite Benutzerbestätigung
vor der Ausführung — das Human-in-the-Loop-Muster aus dem LangGraph-Notebook.

---
# Teil 2 – MCP-Server: Implementierung

## 2.1 Architekturentscheidung: HTTP-basierter MCP-Server

MCP unterstützt zwei Transportschichten:

| Transport | Verwendung | Vorteil |
|---|---|---|
| **stdio** | Lokale Prozesse (Claude Desktop, CLI) | Einfach, kein Netzwerk |
| **HTTP/SSE** | Remote-Server, Microservices, IIoT | Skalierbar, netzwerkfähig |

Für industrielle Systeme ist **HTTP/SSE** (Server-Sent Events) die relevante Variante:
OPC UA-Server, MES-Systeme und Datenbanken laufen typischerweise als Netzwerkdienste.

Wir implementieren den Server mit **FastAPI** und dem `mcp`-SDK.
Dies entspricht dem Produktivmuster für einen MCP-Server, der einen OPC UA-Endpunkt kapselt.

## 2.2 Server-Implementierung

In [ ]:
# ── Maschinenregister: simulierter OPC UA-Adressraum ─────────────────────────
# In einem Produktivsystem würde dieser Abschnitt durch asyncua-Aufrufe ersetzt:
#   from asyncua import Client
#   async with Client("opc.tcp://localhost:4840") as client:
#       node = client.get_node("ns=2;i=1001")
#       temp = await node.read_value()

MASCHINENREGISTER: Dict[str, Dict[str, Any]] = {
    f"M{i+1:02d}": {"temperature": 20.0, "state": "OK", "repair_count": 0}
    for i in range(5)
}
SCHWELLENWERT_HOT  = 70.0
SCHWELLENWERT_WARM = 52.5   # 0.75 × 70
MCP_SERVER_PORT    = 8766


# ── FastAPI-Anwendung als MCP-Server-Stub ────────────────────────────────────
# Hinweis: Das offizielle mcp-SDK bietet eine höherstufige API (mcp.server.Server).
# Hier implementieren wir die JSON-RPC-Schicht manuell, um das Protokoll
# transparent zu machen. Im Produktiveinsatz würde man das SDK verwenden.

mcp_app = FastAPI(title="MCP-Server: Fabriktelemetrie")


# ── Tool-Definitionen (Schema) ────────────────────────────────────────────────
TOOL_DEFINITIONEN = [
    {
        "name": "list_machines",
        "description": "Gibt eine Liste aller Maschinen mit aktuellem Zustand zurück.",
        "inputSchema": {"type": "object", "properties": {}}
    },
    {
        "name": "read_temperature",
        "description": (
            "Liest die aktuelle Temperatur einer Maschine in °C. "
            "Gibt auch den klassifizierten Zustand (OK/WARM/HOT) zurück."
        ),
        "inputSchema": {
            "type": "object",
            "properties": {
                "machine_id": {"type": "string", "description": "Maschinenkennung, z. B. 'M01'"}
            },
            "required": ["machine_id"]
        }
    },
    {
        "name": "write_temperature",
        "description": (
            "Setzt die Temperatur einer Maschine (z. B. nach Reparatur). "
            "Entspricht einem OPC UA write_value()-Aufruf."
        ),
        "inputSchema": {
            "type": "object",
            "properties": {
                "machine_id": {"type": "string"},
                "temperature": {"type": "number", "description": "Neuer Temperaturwert in °C"}
            },
            "required": ["machine_id", "temperature"]
        }
    },
    {
        "name": "simulate_tick",
        "description": (
            "Simuliert einen Zeitschritt: Alle Maschinen erhalten einen zufälligen "
            "Temperaturdrift. Entspricht dem Ablauf eines MESA-Simulationsschritts."
        ),
        "inputSchema": {"type": "object", "properties": {}}
    }
]


# ── Tool-Ausführungslogik ─────────────────────────────────────────────────────
def klassifiziere_zustand(temp: float) -> str:
    if temp > SCHWELLENWERT_HOT:
        return "HOT"
    elif temp > SCHWELLENWERT_WARM:
        return "WARM"
    return "OK"


def fuehre_tool_aus(name: str, argumente: dict) -> dict:
    """Dispatcht einen Tool-Aufruf an die entsprechende Implementierung."""

    if name == "list_machines":
        ergebnis = []
        for mid, daten in MASCHINENREGISTER.items():
            ergebnis.append({
                "machine_id": mid,
                "temperature": daten["temperature"],
                "state": klassifiziere_zustand(daten["temperature"]),
                "repair_count": daten["repair_count"]
            })
        return {"machines": ergebnis}

    elif name == "read_temperature":
        mid = argumente["machine_id"]
        if mid not in MASCHINENREGISTER:
            return {"error": f"Maschine '{mid}' nicht gefunden."}
        temp = MASCHINENREGISTER[mid]["temperature"]
        return {
            "machine_id": mid,
            "temperature": round(temp, 1),
            "unit": "°C",
            "state": klassifiziere_zustand(temp)
        }

    elif name == "write_temperature":
        mid = argumente["machine_id"]
        neuer_wert = float(argumente["temperature"])
        if mid not in MASCHINENREGISTER:
            return {"error": f"Maschine '{mid}' nicht gefunden."}
        MASCHINENREGISTER[mid]["temperature"] = neuer_wert
        if neuer_wert <= 20.0:
            MASCHINENREGISTER[mid]["repair_count"] += 1
        return {"machine_id": mid, "temperature": neuer_wert, "status": "geschrieben"}

    elif name == "simulate_tick":
        aktualisiert = []
        for mid, daten in MASCHINENREGISTER.items():
            temp = daten["temperature"]
            if klassifiziere_zustand(temp) != "HOT":
                temp += random.uniform(-1.0, 5.0)
            MASCHINENREGISTER[mid]["temperature"] = round(temp, 1)
            aktualisiert.append({"machine_id": mid, "temperature": round(temp, 1),
                                  "state": klassifiziere_zustand(temp)})
        return {"tick": "ausgeführt", "maschinen": aktualisiert}

    else:
        return {"error": f"Unbekanntes Tool: '{name}'"}


# ── JSON-RPC 2.0 Endpunkt ─────────────────────────────────────────────────────
class JsonRpcAnfrage(BaseModel):
    jsonrpc: str = "2.0"
    method: str
    params: dict = {}
    id: Optional[int] = None


@mcp_app.post("/mcp")
def mcp_endpunkt(anfrage: JsonRpcAnfrage):
    """Zentraler MCP-Endpunkt. Verarbeitet alle JSON-RPC 2.0-Nachrichten."""

    if anfrage.method == "initialize":
        return {
            "jsonrpc": "2.0",
            "result": {
                "protocolVersion": "2024-11-05",
                "capabilities": {"tools": {}},
                "serverInfo": {"name": "fabrik-mcp-server", "version": "1.0.0"}
            },
            "id": anfrage.id
        }

    elif anfrage.method == "tools/list":
        return {
            "jsonrpc": "2.0",
            "result": {"tools": TOOL_DEFINITIONEN},
            "id": anfrage.id
        }

    elif anfrage.method == "tools/call":
        tool_name = anfrage.params.get("name", "")
        argumente  = anfrage.params.get("arguments", {})
        ergebnis   = fuehre_tool_aus(tool_name, argumente)
        return {
            "jsonrpc": "2.0",
            "result": {
                "content": [{"type": "text", "text": json.dumps(ergebnis, ensure_ascii=False)}]
            },
            "id": anfrage.id
        }

    else:
        return {
            "jsonrpc": "2.0",
            "error": {"code": -32601, "message": f"Methode nicht gefunden: {anfrage.method}"},
            "id": anfrage.id
        }


# ── Server starten ────────────────────────────────────────────────────────────
server_config = uvicorn.Config(
    mcp_app, host="127.0.0.1", port=MCP_SERVER_PORT, log_level="warning"
)
mcp_server = uvicorn.Server(server_config)
mcp_thread = threading.Thread(target=mcp_server.run, daemon=True)
mcp_thread.start()
time.sleep(1.5)

# Verbindungstest
test = requests.post(
    f"http://127.0.0.1:{MCP_SERVER_PORT}/mcp",
    json={"jsonrpc": "2.0", "method": "initialize", "params": {}, "id": 0}
)
print("MCP-Server gestartet. initialize-Antwort:")
print(json.dumps(test.json(), indent=2, ensure_ascii=False))

---
# Teil 3 – MCP-Client: Tool-Discovery und Tool-Call

## 3.1 Client-Klasse

Der MCP-Client kapselt die HTTP-Kommunikation mit dem MCP-Server.
In einer Produktionsumgebung würde man die `mcp`-SDK-Klasse `ClientSession` verwenden;
hier implementieren wir einen einfachen HTTP-Client, um das Protokoll transparent zu halten.

In [ ]:
class MCPClient:
    """Einfacher MCP-Client für HTTP-Transport.
    
    Kapselt JSON-RPC 2.0-Kommunikation mit einem MCP-Server.
    Im Produktiveinsatz: mcp.ClientSession mit stdio- oder SSE-Transport.
    """

    def __init__(self, server_url: str):
        self.server_url = server_url
        self._naechste_id = 1
        self.verfuegbare_tools: List[dict] = []

    def _naechste_anfrage_id(self) -> int:
        id_ = self._naechste_id
        self._naechste_id += 1
        return id_

    def _sende(self, method: str, params: dict) -> dict:
        """Sendet eine JSON-RPC 2.0-Anfrage und gibt das Ergebnis zurück."""
        anfrage = {
            "jsonrpc": "2.0",
            "method": method,
            "params": params,
            "id": self._naechste_anfrage_id()
        }
        antwort = requests.post(self.server_url, json=anfrage, timeout=5.0)
        antwort.raise_for_status()
        daten = antwort.json()
        if "error" in daten:
            raise RuntimeError(f"MCP-Fehler: {daten['error']}")
        return daten.get("result", {})

    def initialisieren(self) -> dict:
        """Handshake mit dem MCP-Server."""
        return self._sende("initialize", {})

    def tools_auflisten(self) -> List[dict]:
        """Fragt alle verfügbaren Tools vom Server ab (Tool-Discovery)."""
        ergebnis = self._sende("tools/list", {})
        self.verfuegbare_tools = ergebnis.get("tools", [])
        return self.verfuegbare_tools

    def tool_aufrufen(self, name: str, argumente: dict = None) -> Any:
        """Ruft ein Tool auf dem Server auf und gibt das geparste Ergebnis zurück."""
        ergebnis = self._sende(
            "tools/call",
            {"name": name, "arguments": argumente or {}}
        )
        # MCP-Antwort: content[0].text enthält das JSON-serialisierte Ergebnis
        content = ergebnis.get("content", [])
        if content and content[0].get("type") == "text":
            return json.loads(content[0]["text"])
        return ergebnis

    def tool_schema_ausgeben(self):
        """Gibt alle Tool-Definitionen leserlich aus."""
        if not self.verfuegbare_tools:
            self.tools_auflisten()
        for tool in self.verfuegbare_tools:
            print(f"\nTool: {tool['name']}")
            print(f"  Beschreibung: {tool['description']}")
            schema = tool.get("inputSchema", {}).get("properties", {})
            if schema:
                print(f"  Parameter:")
                for param, info in schema.items():
                    print(f"    - {param} ({info.get('type','?')}): {info.get('description','')}")


print("MCPClient-Klasse definiert.")

In [ ]:
# ── Client instanziieren und Tool-Discovery durchführen ───────────────────────
client = MCPClient(f"http://127.0.0.1:{MCP_SERVER_PORT}/mcp")

print("=== MCP-Client: Tool-Discovery ===")
client.initialisieren()
client.tool_schema_ausgeben()

print("\n=== Manuelle Tool-Aufrufe ===")

# Alle Maschinen auflisten
maschinen = client.tool_aufrufen("list_machines")
print("\nlist_machines():")
for m in maschinen["machines"]:
    print(f"  {m['machine_id']}: {m['temperature']} °C | {m['state']}")

# Einen Zeitschritt simulieren
tick = client.tool_aufrufen("simulate_tick")
print("\nsimulate_tick():")
for m in tick["maschinen"]:
    print(f"  {m['machine_id']}: {m['temperature']} °C | {m['state']}")

# Einzelne Maschine lesen
ablesen = client.tool_aufrufen("read_temperature", {"machine_id": "M01"})
print(f"\nread_temperature(M01): {ablesen}")

---
# Teil 4 – LLM als Entscheidungsinstanz

## 4.1 Architektur: LLM als CoordinatorAgent

In den bisherigen Notebooks traf der `coordinator_node` Entscheidungen nach
**hartcodierten Regeln**: HOT → Reparatur einplanen.

Ein LLM als Koordinator kann dagegen:
- Kontextuelles Wissen einbringen (z. B. Produktionsplan, Maschinenwichtigkeit)
- Mehrere Faktoren **abwägen** (Temperatur + Trend + Reparaturhistorie)
- Entscheidungen in **natürlicher Sprache begründen** (Auditierbarkeit)
- Auf **unvorhergesehene Situationen** reagieren, die nicht regelbasiert abgedeckt sind

```
MCP-Server                    LLM-Koordinator
(Telemetrie)                  (Entscheidungslogik)
     │                               │
     │  Tool: read_temperature()     │
     │◄──────────────────────────────│
     │  {"M01": 74.2, "M03": 71.1}   │
     │──────────────────────────────►│
     │                               │  Reasoning:
     │                               │  "M01 und M03 sind HOT.
     │                               │   M01 hat höhere Temperatur
     │  Tool: write_temperature()    │   → M01 zuerst reparieren."
     │◄──────────────────────────────│
     │  {"M01": 20.0}                │
     │──────────────────────────────►│
```

## 4.2 Stub-Implementierung (kein API-Key erforderlich)

Der Stub simuliert das Verhalten eines LLMs deterministisch.
Er gibt strukturierte JSON-Antworten zurück, die den realen LLM-Output imitieren.
Dies ermöglicht das Testen der Gesamtarchitektur ohne API-Zugang.

In [ ]:
class LLMKoordinatorStub:
    """Deterministischer LLM-Stub für Tests ohne API-Key.
    
    Imitiert das Verhalten eines echten LLMs:
    - Empfängt einen Prompt mit Maschinentelemetrie
    - Gibt eine strukturierte JSON-Entscheidung zurück
    - Fügt eine Begründung in natürlicher Sprache hinzu
    
    Die Entscheidungslogik ist absichtlich etwas komplexer als
    die regelbasierte Variante: Priorisierung nach Temperatur,
    Berücksichtigung von WARM-Maschinen als Vorwarnstufe.
    """

    def entscheiden(self, maschinentelemetrie: List[dict]) -> dict:
        """Simuliert einen LLM-Aufruf mit Maschinentelemetrie als Kontext.
        
        Rückgabe:
          {
            "sofortiger_eingriff": ["M01", "M03"],  # HOT-Maschinen, absteigend nach Temp
            "beobachtung":         ["M02"],          # WARM-Maschinen
            "begruendung":         "...",            # Reasoning in natürlicher Sprache
            "risikobewertung":     "hoch"            # niedrig / mittel / hoch
          }
        """
        hot_maschinen  = [(m["machine_id"], m["temperature"])
                          for m in maschinentelemetrie if m["state"] == "HOT"]
        warm_maschinen = [m["machine_id"]
                          for m in maschinentelemetrie if m["state"] == "WARM"]

        # LLM-typisches Verhalten: nach Temperatur absteigend priorisieren
        hot_maschinen.sort(key=lambda x: x[1], reverse=True)
        sofortiger_eingriff = [mid for mid, _ in hot_maschinen]

        # Begründung generieren (simuliert LLM-Output)
        if hot_maschinen:
            hot_liste = ", ".join(
                f"{mid} ({temp:.1f} °C)" for mid, temp in hot_maschinen
            )
            begruendung = (
                f"Kritische Überhitzung festgestellt: {hot_liste}. "
                f"Reparatur in absteigender Temperaturpriorität empfohlen. "
            )
            if warm_maschinen:
                begruendung += (
                    f"Maschinen {', '.join(warm_maschinen)} im Warnbereich — "
                    f"engmaschige Überwachung empfohlen."
                )
            risiko = "hoch" if len(hot_maschinen) >= 2 else "mittel"
        elif warm_maschinen:
            begruendung = (
                f"Keine kritischen Maschinen. {', '.join(warm_maschinen)} im "
                f"Warnbereich — Trend beobachten."
            )
            risiko = "niedrig"
        else:
            begruendung = "Alle Maschinen im Normalbetrieb. Keine Maßnahmen erforderlich."
            risiko = "niedrig"

        return {
            "sofortiger_eingriff": sofortiger_eingriff,
            "beobachtung": warm_maschinen,
            "begruendung": begruendung,
            "risikobewertung": risiko
        }


# ── Stub testen ───────────────────────────────────────────────────────────────
stub = LLMKoordinatorStub()

test_telemetrie = [
    {"machine_id": "M01", "temperature": 78.3, "state": "HOT"},
    {"machine_id": "M02", "temperature": 55.1, "state": "WARM"},
    {"machine_id": "M03", "temperature": 74.5, "state": "HOT"},
    {"machine_id": "M04", "temperature": 32.0, "state": "OK"},
    {"machine_id": "M05", "temperature": 22.1, "state": "OK"},
]

entscheidung = stub.entscheiden(test_telemetrie)
print("=== LLM-Koordinator-Stub: Entscheidung ===")
print(json.dumps(entscheidung, indent=2, ensure_ascii=False))

## 4.3 Echter LLM-API-Aufruf (Erweiterung)

Der folgende Abschnitt ist **auskommentiert** und erfordert einen Anthropic-API-Key
(alternativ: OpenAI-API-Key, dann `ChatOpenAI` statt `ChatAnthropic` verwenden).

Die Schnittstelle zur restlichen Implementierung ist **identisch** zum Stub:
`entscheiden(maschinentelemetrie) → dict`. Der Rest des Graphen (Teil 5)
muss nicht verändert werden.

In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
# ERWEITERUNG: Echter LLM-API-Aufruf (API-Key erforderlich)
# Zum Aktivieren: Kommentarzeichen entfernen und API_KEY setzen
# ════════════════════════════════════════════════════════════════════════════════

# import os
# from langchain_anthropic import ChatAnthropic
# from langchain_core.messages import HumanMessage, SystemMessage

# ANTHROPIC_API_KEY = os.getenv("ANTHROPIC_API_KEY", "sk-ant-...")

# class LLMKoordinatorEcht:
#     """Echter LLM-Koordinator via Anthropic Claude API.
#     
#     Schnittstelle identisch zum Stub → Direkter Austausch möglich.
#     """
#
#     SYSTEM_PROMPT = textwrap.dedent("""
#         Sie sind ein KI-Koordinatoragent in einer Fertigungsanlage.
#         Ihre Aufgabe: Maschinentelemetrie analysieren und Wartungsmaßnahmen priorisieren.
#
#         Schwellenwerte:
#         - HOT:  Temperatur > 70 °C → sofortige Reparatur erforderlich
#         - WARM: Temperatur > 52,5 °C → Beobachtung und Vorwarnung
#         - OK:   Temperatur ≤ 52,5 °C → kein Handlungsbedarf
#
#         Antworten Sie AUSSCHLIESSLICH im folgenden JSON-Format:
#         {
#             "sofortiger_eingriff": ["M01", ...],
#             "beobachtung": ["M02", ...],
#             "begruendung": "...",
#             "risikobewertung": "niedrig" | "mittel" | "hoch"
#         }
#         Kein Text vor oder nach dem JSON. Kein Markdown.
#     """)
#
#     def __init__(self, api_key: str):
#         self.llm = ChatAnthropic(
#             model="claude-sonnet-4-20250514",
#             api_key=api_key,
#             max_tokens=512,
#             temperature=0.1   # niedrige Temperatur für konsistente JSON-Ausgabe
#         )
#
#     def entscheiden(self, maschinentelemetrie: List[dict]) -> dict:
#         """Ruft das LLM mit aktueller Maschinentelemetrie auf."""
#         telemetrie_text = json.dumps(maschinentelemetrie, indent=2, ensure_ascii=False)
#         user_prompt = f"Aktuelle Maschinentelemetrie:\n{telemetrie_text}\n\nEntscheidung:"
#
#         nachrichten = [
#             SystemMessage(content=self.SYSTEM_PROMPT),
#             HumanMessage(content=user_prompt)
#         ]
#
#         antwort = self.llm.invoke(nachrichten)
#         rohtext = antwort.content.strip()
#
#         # JSON sicher parsen
#         try:
#             return json.loads(rohtext)
#         except json.JSONDecodeError:
#             # Fallback: Stub verwenden, wenn JSON-Parsing fehlschlägt
#             print(f"[WARNUNG] LLM-Ausgabe kein valides JSON: {rohtext[:100]}")
#             return LLMKoordinatorStub().entscheiden(maschinentelemetrie)
#
# # Instanziierung:
# # koordinator_llm = LLMKoordinatorEcht(api_key=ANTHROPIC_API_KEY)
# # entscheidung = koordinator_llm.entscheiden(test_telemetrie)
# # print(json.dumps(entscheidung, indent=2, ensure_ascii=False))

print("Echter LLM-Aufruf: Code bereit, auskommentiert (kein API-Key erforderlich).")
print("Zum Aktivieren: Kommentarzeichen entfernen und ANTHROPIC_API_KEY setzen.")

---
# Teil 5 – Vollintegration: LangGraph + MCP + LLM

## 5.1 Systemarchitektur

```
┌──────────────────────────────────────────────────────────────────────┐
│  LangGraph Host                                                      │
│                                                                      │
│  START → [machine_node] → [coordinator_node] → [maintenance_node]   │
│               │                   │                    │             │
│               │ MCP:              │ LLM-Stub /         │ MCP:        │
│               │ simulate_tick()   │ ChatAnthropic       │ write_temp()│
│               │ read_all()        │                    │             │
│               ▼                   ▼                    ▼             │
│         MCP-Server           Entscheidung         MCP-Server        │
│         (Telemetrie)         (JSON)               (Aktuator)        │
└──────────────────────────────────────────────────────────────────────┘
```

## 5.2 Erweiterter GraphState

In [ ]:
class VollGraphState(TypedDict):
    """Erweiterter Zustand für die Vollintegration.
    
    Ergänzungen gegenüber dem LangGraph-Notebook:
    - llm_begruendung: Reasoning-Ausgabe des LLM-Koordinators
    - risikobewertung: LLM-Einschätzung des Gesamtrisikos
    - beobachtungsliste: WARM-Maschinen (LLM-Vorwarnung)
    - mcp_log: Protokoll aller MCP-Tool-Aufrufe
    """
    # Maschinenzustände (aus MCP-Server)
    telemetrie: List[dict]            # Rohdaten vom MCP-Server

    # Koordinatorentscheidungen (LLM-Ausgabe)
    repair_queue: List[str]
    beobachtungsliste: List[str]      # NEU: WARM-Maschinen
    llm_begruendung: str              # NEU: LLM-Reasoning
    risikobewertung: str              # NEU: niedrig / mittel / hoch
    coordinator_log: List[str]

    # Wartungsprotokoll
    maintenance_log: List[str]

    # MCP-Protokoll
    mcp_log: List[str]                # NEU: alle Tool-Aufrufe

    # Steuerung
    cycle: int
    max_cycles: int
    next_action: str


def voll_initial_state(max_cycles: int = 8) -> VollGraphState:
    return VollGraphState(
        telemetrie=[],
        repair_queue=[], beobachtungsliste=[],
        llm_begruendung="", risikobewertung="unbekannt",
        coordinator_log=[], maintenance_log=[], mcp_log=[],
        cycle=0, max_cycles=max_cycles, next_action="machine_node",
    )

print("VollGraphState definiert.")

In [ ]:
# ── Koordinator-Instanz (Stub oder echt) ──────────────────────────────────────
# Hier den Stub verwenden; für echten API-Aufruf durch LLMKoordinatorEcht ersetzen:
# koordinator = LLMKoordinatorEcht(api_key=ANTHROPIC_API_KEY)
koordinator = LLMKoordinatorStub()

# MCP-Client-Instanz
mcp = MCPClient(f"http://127.0.0.1:{MCP_SERVER_PORT}/mcp")
mcp.initialisieren()


# ── machine_node: Telemetrie via MCP abrufen ──────────────────────────────────
def voll_machine_node(state: VollGraphState) -> dict:
    """Holt Maschinentelemetrie über MCP-Tool-Aufrufe.
    
    MCP-Aufrufe:
      1. simulate_tick()  → Temperaturen aktualisieren (simuliert OPC UA-Zeitreihe)
      2. list_machines()  → Alle Zustände lesen
    """
    zyklus = state["cycle"]
    mcp_log = list(state["mcp_log"])

    # 1. Zeitschritt simulieren (entspricht: OPC UA-Werte im PLC aktualisieren)
    mcp.tool_aufrufen("simulate_tick")
    mcp_log.append(f"[Zyklus {zyklus}] MCP: simulate_tick() ausgeführt")

    # 2. Telemetrie lesen
    ergebnis = mcp.tool_aufrufen("list_machines")
    telemetrie = ergebnis["machines"]
    mcp_log.append(f"[Zyklus {zyklus}] MCP: list_machines() → {len(telemetrie)} Maschinen")

    # Konsolenausgabe
    print(f"  [MachineNode] Zyklus {zyklus}: Telemetrie abgerufen")
    for m in telemetrie:
        marker = " ⚠" if m["state"] == "HOT" else (" ○" if m["state"] == "WARM" else "")
        print(f"    {m['machine_id']}: {m['temperature']} °C [{m['state']}]{marker}")

    return {"telemetrie": telemetrie, "mcp_log": mcp_log, "next_action": "coordinator_node"}


# ── coordinator_node: LLM-Entscheidung ───────────────────────────────────────
def voll_coordinator_node(state: VollGraphState) -> dict:
    """LLM-Koordinator: analysiert Telemetrie und trifft Reparaturentscheidung.
    
    In diesem Knoten wird der LLM-Aufruf (Stub oder echt) durchgeführt.
    Das Ergebnis enthält priorisierten Reparaturplan + Begründung.
    """
    zyklus = state["cycle"]
    protokoll = list(state["coordinator_log"])

    # LLM-Entscheidung abrufen
    entscheidung = koordinator.entscheiden(state["telemetrie"])

    repair_queue     = entscheidung.get("sofortiger_eingriff", [])
    beobachtung      = entscheidung.get("beobachtung", [])
    begruendung      = entscheidung.get("begruendung", "")
    risikobewertung  = entscheidung.get("risikobewertung", "unbekannt")

    protokoll.append(
        f"[Zyklus {zyklus}] [LLM-KOORD] Risiko: {risikobewertung} | "
        f"Reparatur: {repair_queue} | Beobachtung: {beobachtung}"
    )
    protokoll.append(f"[Zyklus {zyklus}] [LLM-KOORD] Begründung: {begruendung}")

    print(f"  [CoordinatorNode] LLM-Entscheidung: Risiko={risikobewertung}")
    print(f"    Reparatur-Queue: {repair_queue}")
    print(f"    Begründung: {begruendung[:80]}..." if len(begruendung) > 80 else
          f"    Begründung: {begruendung}")

    naechste_aktion = "maintenance_node" if repair_queue else "advance_cycle"

    return {
        "repair_queue": repair_queue,
        "beobachtungsliste": beobachtung,
        "llm_begruendung": begruendung,
        "risikobewertung": risikobewertung,
        "coordinator_log": protokoll,
        "next_action": naechste_aktion,
    }


# ── maintenance_node: Reparatur via MCP schreiben ─────────────────────────────
def voll_maintenance_node(state: VollGraphState) -> dict:
    """Repariert erste Maschine in der Queue und schreibt Reset via MCP.
    
    MCP-Aufruf:
      write_temperature(machine_id, 20.0) → entspricht OPC UA write_value()
    """
    queue    = list(state["repair_queue"])
    protokoll = list(state["maintenance_log"])
    mcp_log  = list(state["mcp_log"])
    zyklus   = state["cycle"]

    if not queue:
        return {"next_action": "advance_cycle"}

    ziel = queue.pop(0)

    # Reparatur via MCP-Tool schreiben
    ergebnis = mcp.tool_aufrufen("write_temperature", {"machine_id": ziel, "temperature": 20.0})
    mcp_log.append(
        f"[Zyklus {zyklus}] MCP: write_temperature({ziel}, 20.0) → {ergebnis.get('status','?')}"
    )

    eintrag = f"[Zyklus {zyklus}] [WARTUNG] {ziel} repariert (MCP-Schreibzugriff bestätigt)."
    protokoll.append(eintrag)
    print(f"  [MaintenanceNode] {eintrag}")

    return {
        "repair_queue": queue,
        "maintenance_log": protokoll,
        "mcp_log": mcp_log,
        "next_action": "advance_cycle",
    }


def voll_advance_cycle(state: VollGraphState) -> dict:
    print(f"\n─── Ende von Zyklus {state['cycle']} ───\n")
    return {"cycle": state["cycle"] + 1, "next_action": "machine_node"}


print("Alle Knoten für Vollintegration definiert.")

In [ ]:
# ── Graphen aufbauen ──────────────────────────────────────────────────────────
def voll_router(state: VollGraphState):
    if state["cycle"] >= state["max_cycles"]:
        return "__end__"
    return state["next_action"]


voll_builder = StateGraph(VollGraphState)
voll_builder.add_node("machine_node",     voll_machine_node)
voll_builder.add_node("coordinator_node", voll_coordinator_node)
voll_builder.add_node("maintenance_node", voll_maintenance_node)
voll_builder.add_node("advance_cycle",    voll_advance_cycle)
voll_builder.set_entry_point("machine_node")

ziele_voll = {
    "machine_node":     "machine_node",
    "coordinator_node": "coordinator_node",
    "maintenance_node": "maintenance_node",
    "advance_cycle":    "advance_cycle",
    "__end__":          END,
}
for knoten in ["machine_node", "coordinator_node", "maintenance_node", "advance_cycle"]:
    voll_builder.add_conditional_edges(knoten, voll_router, ziele_voll)

voll_graph = voll_builder.compile()

print("=" * 60)
print("VOLLINTEGRATION: LangGraph + MCP + LLM")
print("=" * 60 + "\n")

voll_endzustand = voll_graph.invoke(voll_initial_state(max_cycles=60))

print("\n" + "=" * 60)
print("SIMULATION ABGESCHLOSSEN")
print("=" * 60)

In [ ]:
# ── Auswertung ────────────────────────────────────────────────────────────────
print("\n=== LLM-Koordinator-Protokoll ===")
for e in voll_endzustand["coordinator_log"]:
    print(" ", e)

print("\n=== Wartungsprotokoll ===")
for e in voll_endzustand["maintenance_log"]:
    print(" ", e)

print("\n=== MCP-Tool-Protokoll ===")
for e in voll_endzustand["mcp_log"]:
    print(" ", e)

print("\n=== Abschlusszustand auf dem MCP-Server ===")
finale_maschinen = mcp.tool_aufrufen("list_machines")
print(f"{'Maschine':<10} {'Temperatur':>13} {'Zustand':<8} {'Reparaturen'}")
print("-" * 48)
for m in finale_maschinen["machines"]:
    print(f"{m['machine_id']:<10} {m['temperature']:>10.1f} °C "
          f"{m['state']:<8} {m['repair_count']}×")

---
# Teil 6 – Vergleich: Regelbasiert vs. LLM-gestützt

## 6.1 Systematischer Vergleich

In [ ]:
# Ausgabe als strukturierte Tabelle im Notebook
vergleich = [
    ("Entscheidungsgrundlage",
     "Hartcodierte Schwellenwerte (temp > 70 °C)",
     "Kontextuelles Reasoning über alle Faktoren"),
    ("Priorisierung",
     "FIFO (Eingangsreihenfolge in die Queue)",
     "Temperaturabsteigend + kontextuell"),
    ("Begründbarkeit",
     "Implizit (Code = Logik)",
     "Explizit (natürlichsprachliche Begründung)"),
    ("Unbekannte Situationen",
     "Nicht abgedeckt → kein Eingriff",
     "Analogieschluss aus Trainingsdaten"),
    ("Deterministik",
     "Vollständig deterministisch",
     "Nicht-deterministisch (temperature > 0)"),
    ("Latenz pro Entscheidung",
     "< 1 ms",
     "500 ms – 5 s (API-Roundtrip)"),
    ("Kosten",
     "Keine API-Kosten",
     "Token-basierte API-Kosten"),
    ("Anpassbarkeit",
     "Code-Änderung erforderlich",
     "Prompt-Engineering genügt"),
    ("Auditierbarkeit",
     "Vollständig (deterministisch nachvollziehbar)",
     "Teilweise (Reasoning-Text, aber nicht formal verifikabel)"),
    ("Eignung",
     "Echtzeit-Regelung, Safety-kritische Systeme",
     "Diagnose, Priorisierung, Erklärung"),
]

print(f"{'Dimension':<28} {'Regelbasiert':<44} {'LLM-gestützt'}")
print("─" * 120)
for dim, regel, llm in vergleich:
    print(f"{dim:<28} {regel:<44} {llm}")

## 6.2 Empfehlung: Hybridarchitektur

Für industrielle Produktivsysteme empfiehlt sich eine **zweischichtige Architektur**:

```
Schicht 1 – Echtzeit-Regelung (regelbasiert, < 1 ms)
─────────────────────────────────────────────────────
  temp > 80 °C → Notabschaltung (SPS-Ebene, kein LLM)
  temp > 70 °C → HOT-Flag setzen (OPC UA)

Schicht 2 – Orchestrierung & Diagnose (LLM-gestützt, Sekunden)
──────────────────────────────────────────────────────────────
  Reparaturpriorisierung, Trendanalyse, Begründung
  → LangGraph + MCP + LLM (dieses Notebook)

Schicht 3 – Strategische Planung (LLM-gestützt, Minuten–Stunden)
─────────────────────────────────────────────────────────────────
  Wartungsplanung, Kapazitätsoptimierung, Reporting
```

**Kernprinzip**: LLMs ersetzen nicht die deterministischen Steuerungssysteme
der Feldebene — sie ergänzen sie auf der Orchestrierungsebene.
Sicherheitskritische Entscheidungen (Notabschaltung, Verriegelung) verbleiben
immer in deterministischen Regelwerken.

## 6.3 MCP als Enabler: Warum MCP diese Architektur ermöglicht

Ohne MCP müsste jede der drei Schichten eigene OPC UA-Clients, REST-Adapter
und Datenbankverbindungen verwalten — ein Wartungsaufwand, der in industriellen
Umgebungen prohibitiv ist.

MCP entkoppelt:
- **Was** abgerufen wird (Tool-Name + Parameter) von
- **Wie** es abgerufen wird (OPC UA, REST, SQL, MQTT)

Ein OPC UA-MCP-Server kann von LangGraph-Knoten, n8n-Workflows und
zukünftigen A2A-Agenten gleichermaßen genutzt werden — ohne Codeänderung
auf der Konsumentenseite.

---
## Zusammenfassung

| Konzept | Implementierung in diesem Notebook | Produktivvariante |
|---|---|---|
| **MCP-Server** | FastAPI + manuelles JSON-RPC | `mcp.server.Server` + asyncua |
| **MCP-Client** | `MCPClient`-Klasse mit `requests` | `mcp.ClientSession` (SDK) |
| **Tool-Discovery** | `tools/list` → JSON-Schema | Automatisch via SDK |
| **LLM-Koordinator** | `LLMKoordinatorStub` | `ChatAnthropic` / `ChatOpenAI` |
| **Structured Output** | JSON-Prompt + `json.loads()` | Pydantic-Schema + LangChain |
| **LangGraph-Integration** | Knoten ruft MCP-Client auf | Identisch |
| **Transport** | HTTP/REST | HTTP/SSE (MCP-Standard) oder stdio |

---

**Weiterführende Ressourcen**

- MCP-Spezifikation: https://modelcontextprotocol.io/specification
- MCP Python SDK: https://github.com/modelcontextprotocol/python-sdk
- LangChain MCP-Integration: https://python.langchain.com/docs/integrations/tools/mcp
- Anthropic Claude API: https://docs.anthropic.com/en/api
- LangGraph-Dokumentation: https://langchain-ai.github.io/langgraph/